# 28.07 - Class-imbalance training: weighted loss

**Notebook type:** Solution.

**Daily output:** Weighted-loss training comparison notebook using a harder imbalanced image dataset.

You will compare ordinary cross-entropy with class-weighted cross-entropy while holding the split, seed, model initialization, loader order, and evaluation distribution constant.


## Core Ideas

- Calculate class weights from the training split only.
- `compute_class_weight(class_weight="balanced", ...)` returns one weight per class in the supplied class order.
- `nn.CrossEntropyLoss(weight=...)` expects a floating tensor whose index matches the model's logit column.
- Keep validation unweighted and in its natural distribution.
- Weighted loss changes optimization, not the observed training sample distribution.
- Evaluate accuracy, Macro-F1, per-class recall, confusion matrices, and learning curves together.


In [ ]:
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, recall_score

SEED = 28
NUM_CLASSES = 3
np.random.seed(SEED)
torch.manual_seed(SEED)


## Prepared Harder Imbalanced Image Data

The three classes use noisy, shifted, partly occluded spatial patterns. Counts are deliberately imbalanced.

**Return structure — `make_imbalanced_pattern_images`:** Returns a tuple. Position 0 is a CPU `torch.float32` tensor `[N,3,24,24]` with values in `[0,1]`. Position 1 is a CPU `torch.long` tensor `[N]`. `N=sum(counts)` and labels are contiguous from zero.


In [ ]:
def make_imbalanced_pattern_images(counts=(72, 24, 12), image_size=24, seed=28):
    rng = np.random.default_rng(seed)
    images, labels = [], []
    for class_index, class_count in enumerate(counts):
        for sample_index in range(class_count):
            image = rng.normal(0.18, 0.22, size=(3, image_size, image_size)).astype(np.float32)
            shift = int(rng.integers(-3, 4))
            main_channel = int(rng.integers(0, 3))
            detail_channel = (main_channel + int(rng.integers(1, 3))) % 3
            if class_index in (0, 2):
                center = image_size // 2 + shift
                image[main_channel, 3:image_size - 3, center - 2:center + 2] += 0.52
                if class_index == 2:
                    # The rare class shares the majority vertical bar and differs
                    # only by a short, partially noisy cross-piece.
                    detail_row = image_size // 2 + int(rng.integers(-3, 4))
                    image[detail_channel, detail_row - 1:detail_row + 2, center - 6:center + 7] += 0.46
            elif class_index == 1:
                center = image_size // 2 + shift
                image[main_channel, center - 2:center + 2, 3:image_size - 3] += 0.52
            # Every class receives an unrelated patch and occasional occlusion.
            patch_top = int(rng.integers(2, image_size - 6))
            patch_left = int(rng.integers(2, image_size - 6))
            image[detail_channel, patch_top:patch_top + 4, patch_left:patch_left + 4] += 0.30
            if sample_index % 3 == 0:
                top = int(rng.integers(4, image_size - 8))
                left = int(rng.integers(4, image_size - 8))
                image[:, top:top + 5, left:left + 6] *= 0.08
            images.append(np.clip(image, 0.0, 1.0))
            labels.append(class_index)
    order = rng.permutation(len(labels))
    return torch.tensor(np.stack(images)[order], dtype=torch.float32), torch.tensor(np.asarray(labels)[order], dtype=torch.long)


images, labels = make_imbalanced_pattern_images()
print("dataset:", images.shape, images.dtype, torch.bincount(labels).tolist())


## Exercise 28-A: Create a stratified untouched validation split

Use `train_test_split(..., stratify=labels)` on row indices. Normalize with statistics from training images only.

**Return structure — `split_imbalanced_images`:** Returns a `dict` with exactly `train_images`, `val_images` (CPU `torch.float32` tensors `[N_train,3,24,24]` and `[N_val,3,24,24]`); `train_labels`, `val_labels`, `train_indices`, and `val_indices` (CPU `torch.long` tensors); and scalar CPU `torch.float32` tensors `mean` and `std`. Indices are disjoint and cover all source rows.


In [ ]:
def split_imbalanced_images(images, labels, val_fraction=0.25, seed=SEED):
    all_indices = np.arange(len(labels))
    train_array, val_array = train_test_split(
        all_indices,
        test_size=val_fraction,
        random_state=seed,
        stratify=labels.numpy(),
    )
    train_indices = torch.tensor(train_array, dtype=torch.long)
    val_indices = torch.tensor(val_array, dtype=torch.long)
    raw_train = images[train_indices]
    raw_val = images[val_indices]
    mean = raw_train.mean().float()
    std = raw_train.std().float().clamp_min(1e-6)
    return {
        "train_images": ((raw_train - mean) / std).float(),
        "val_images": ((raw_val - mean) / std).float(),
        "train_labels": labels[train_indices].long(),
        "val_labels": labels[val_indices].long(),
        "train_indices": train_indices,
        "val_indices": val_indices,
        "mean": mean,
        "std": std,
    }


# Smoke check
split = split_imbalanced_images(images, labels)
print("train/validation counts:", torch.bincount(split["train_labels"]).tolist(), torch.bincount(split["val_labels"]).tolist())


## Exercise 28-B: Calculate balanced class weights

Use `sklearn.utils.class_weight.compute_class_weight` with explicit class order `0..C-1`.

**Return structure — `calculate_class_weights`:** Returns a CPU `torch.float32` tensor shaped `[num_classes]`. Every value is finite and positive, and tensor index `c` is the loss weight for class `c`.


In [ ]:
def calculate_class_weights(train_labels, num_classes):
    classes = np.arange(num_classes)
    weights = compute_class_weight(
        class_weight="balanced",
        classes=classes,
        y=train_labels.numpy(),
    )
    return torch.tensor(weights, dtype=torch.float32)


# Smoke check
class_weights = calculate_class_weights(split["train_labels"], NUM_CLASSES)
print("class weights:", class_weights.tolist())


## Exercise 28-C: Build a compact CV model

Use convolution, batch normalization, nonlinearities, pooling, and a linear classifier.

**Return structure — `TinyImbalanceCNN`:** A callable `nn.Module`. Construction returns a CPU module. Calling it with a CPU `torch.float32` tensor `[N,3,H,W]` returns CPU `torch.float32` logits `[N,num_classes]`.


In [ ]:
class TinyImbalanceCNN(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 12, kernel_size=3, padding=1),
            nn.BatchNorm2d(12),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(12, 24, kernel_size=3, padding=1),
            nn.BatchNorm2d(24),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d((1, 1)),
        )
        self.classifier = nn.Linear(24, num_classes)

    def forward(self, batch):
        features = self.features(batch).flatten(1)
        return self.classifier(features)


# Smoke check
smoke_model = TinyImbalanceCNN(NUM_CLASSES)
smoke_logits = smoke_model(split["train_images"][:4])
print("CNN logits:", smoke_logits.shape)


## Exercise 28-D: Train and evaluate one controlled run

Accept an optional loss-weight tensor, train for a small number of epochs, and evaluate on the untouched validation loader.

**Return structure — `train_weighted_loss_run`:** Returns a `dict` with exactly `model` (`TinyImbalanceCNN` on CPU), `history` (`list[float]` of length `epochs`), and `metrics` (a dictionary with `accuracy` and `macro_f1` as floats; `per_class_recall` as a CPU `torch.float32` tensor `[C]`; and `confusion_matrix` as a CPU `torch.long` tensor `[C,C]`).


In [ ]:
def train_weighted_loss_run(split, num_classes, loss_weights=None, epochs=4, seed=SEED):
    torch.manual_seed(seed)
    model = TinyImbalanceCNN(num_classes)
    criterion = nn.CrossEntropyLoss(weight=loss_weights)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.012)
    train_loader = DataLoader(
        TensorDataset(split["train_images"], split["train_labels"]),
        batch_size=18,
        shuffle=True,
        generator=torch.Generator().manual_seed(seed),
    )
    val_loader = DataLoader(
        TensorDataset(split["val_images"], split["val_labels"]),
        batch_size=18,
        shuffle=False,
    )
    history = []
    for _ in range(epochs):
        model.train()
        total_loss = 0.0
        total_count = 0
        for batch_images, batch_labels in train_loader:
            optimizer.zero_grad()
            loss = criterion(model(batch_images), batch_labels)
            loss.backward()
            optimizer.step()
            total_loss += loss.item() * len(batch_labels)
            total_count += len(batch_labels)
        history.append(float(total_loss / total_count))
    model.eval()
    prediction_parts, label_parts = [], []
    with torch.inference_mode():
        for batch_images, batch_labels in val_loader:
            prediction_parts.append(model(batch_images).argmax(dim=1).cpu())
            label_parts.append(batch_labels.cpu())
    predictions = torch.cat(prediction_parts).long()
    validation_labels = torch.cat(label_parts).long()
    labels_array = np.arange(num_classes)
    metrics = {
        "accuracy": float(accuracy_score(validation_labels.numpy(), predictions.numpy())),
        "macro_f1": float(f1_score(validation_labels.numpy(), predictions.numpy(), labels=labels_array, average="macro", zero_division=0)),
        "per_class_recall": torch.tensor(recall_score(validation_labels.numpy(), predictions.numpy(), labels=labels_array, average=None, zero_division=0), dtype=torch.float32),
        "confusion_matrix": torch.tensor(confusion_matrix(validation_labels.numpy(), predictions.numpy(), labels=labels_array), dtype=torch.long),
    }
    return {"model": model, "history": history, "metrics": metrics}


# Smoke check
smoke_run = train_weighted_loss_run(split, NUM_CLASSES, loss_weights=None, epochs=1)
print("baseline smoke metrics:", smoke_run["metrics"])


## Exercise 28-E: Compare standard and weighted loss

Run both strategies with the same seed and verify that their models begin from the same initialization before optimization.

**Return structure — `compare_weighted_loss`:** Returns a `dict` with exactly `standard` and `weighted`, each the result of `train_weighted_loss_run`, plus `class_weights`, a CPU `torch.float32` tensor `[C]`, and `minority_recall_change`, a Python `float` equal to weighted minus standard recall for class `C-1`.


In [ ]:
def compare_weighted_loss(split, num_classes, epochs=4, seed=SEED):
    weights = calculate_class_weights(split["train_labels"], num_classes)
    standard = train_weighted_loss_run(split, num_classes, loss_weights=None, epochs=epochs, seed=seed)
    weighted = train_weighted_loss_run(split, num_classes, loss_weights=weights, epochs=epochs, seed=seed)
    change = weighted["metrics"]["per_class_recall"][-1].item() - standard["metrics"]["per_class_recall"][-1].item()
    return {
        "standard": standard,
        "weighted": weighted,
        "class_weights": weights,
        "minority_recall_change": float(change),
    }


# Smoke check
comparison = compare_weighted_loss(split, NUM_CLASSES, epochs=3)
print("standard Macro-F1:", comparison["standard"]["metrics"]["macro_f1"])
print("weighted Macro-F1:", comparison["weighted"]["metrics"]["macro_f1"])
print("minority recall change:", comparison["minority_recall_change"])


## Test Cases

Run this cell after completing all TODO cells. A correct implementation prints `Day 28 tests passed`.

**Return structure — `run_day28_tests`:** Returns `None`. Success is communicated by assertions completing and the exact printed message `Day 28 tests passed`.


In [ ]:
def run_day28_tests():
    expected_keys = {"train_images", "val_images", "train_labels", "val_labels", "train_indices", "val_indices", "mean", "std"}
    assert set(split) == expected_keys
    assert split["train_images"].dtype == torch.float32 and split["train_labels"].dtype == torch.long
    assert set(split["train_indices"].tolist()).isdisjoint(split["val_indices"].tolist())
    assert len(split["train_indices"]) + len(split["val_indices"]) == len(labels)
    assert abs(split["train_images"].mean().item()) < 1e-5

    weights = calculate_class_weights(split["train_labels"], NUM_CLASSES)
    assert weights.shape == (NUM_CLASSES,) and weights.dtype == torch.float32
    assert torch.isfinite(weights).all() and torch.all(weights > 0)
    assert weights[-1] > weights[0]

    logits = TinyImbalanceCNN(NUM_CLASSES)(split["train_images"][:5])
    assert logits.shape == (5, NUM_CLASSES)
    result = comparison
    assert set(result) == {"standard", "weighted", "class_weights", "minority_recall_change"}
    for strategy in ("standard", "weighted"):
        assert len(result[strategy]["history"]) == 3
        metrics = result[strategy]["metrics"]
        assert set(metrics) == {"accuracy", "macro_f1", "per_class_recall", "confusion_matrix"}
        assert metrics["per_class_recall"].shape == (NUM_CLASSES,)
        assert metrics["confusion_matrix"].shape == (NUM_CLASSES, NUM_CLASSES)
        assert int(metrics["confusion_matrix"].sum()) == len(split["val_labels"])
    print("Day 28 tests passed")


run_day28_tests()


## Day 28 Checklist

- [ ] I computed weights from training labels only.
- [ ] I kept validation unweighted and untouched.
- [ ] I preserved class-index order between weights and logits.
- [ ] I compared runs with the same split, seed, and epoch budget.
- [ ] I interpreted Macro-F1 together with per-class recall and the confusion matrix.
